# Smart Medic — RUNBOOK

**Đường thẳng từ repo sạch tới `output.zip`.** Chỉ các lệnh thực thi chính, đúng thứ tự.
Không giải thích thiết kế, không phân tích — những thứ đó ở
[`docs/reports/plan-v4.html`](../docs/reports/plan-v4.html).

Chạy **từ trên xuống**. Mỗi ô in `✓` hoặc `✗`; ô nào `✗` thì **dừng ở đó** — các ô sau không
có nghĩa nữa.

| Ô | Bước | Phase | Bắt buộc |
|---|---|---|---|
| 1 | Môi trường | — | ✓ |
| 2 | **Cổng toàn vẹn** — offset + `data/test/` chưa bị sửa | P0 | ✓ **không được bỏ** |
| 3 | Dựng chỉ mục KB + gazetteer làn R | P1 · P5 | ✓ |
| 4 | Chạy trên **gold** → chấm điểm + lát cắt + bootstrap (ĐO) | P0–P6 · **P2** | ✓ |
| 5 | Chạy trên **`data/test/`** → `data/output/` (NỘP) | P0–P6 | ✓ |
| 6 | Đóng gói `output.zip` + ba zip probe + manifest | P0 · **P2** · P7 | ✓ |
| 7 | Diễn tập tái lập (container sạch) | P7 | ✓ **chống bị loại** |
| 8 | **Tự kiểm** — notebook còn khớp repo không | — | ✓ sau **mỗi** phase |

> 🔁 **File này phải được chạy lại và cập nhật sau MỖI phase.** Mỗi phase thêm file mới và
> đôi khi đổi cờ dòng lệnh; ô 8 tự phát hiện chỗ lệch. Notebook trôi khỏi repo còn tệ hơn
> không có notebook — nó khiến người ta tin vào một lệnh đã sai.

> **Trạng thái hôm nay:** làn R của `extract/` (P1) đã chạy — không model, không GPU, không
> checkpoint. Ô 4 và ô 5 chạy pipeline thật. `assertion/` (P4) và `linking/` (P5) chưa có,
> nên mọi record ra với `assertions` và `candidates` RỖNG; đó là câu trả lời đúng cho hai
> loại xét nghiệm (11,59 điểm) và bằng 0 ở những chỗ khác — đúng con số mà một phỏng đoán
> sai cũng nhận được.
>
> Hệ quả: `data/output/` hiện **chính là Probe A**, nên `output.zip` và
> `runs/p2/output_probe_A.zip` byte-identical. Hạ tầng đo của P2 (`eval/slices.py`,
> `eval/bootstrap.py`, `eval/probe.py`, `tests/test_alignment_parity.py`) đã có ở ô 4 và ô 6.
> **Nộp bài vẫn là quyết định của con người** — notebook dựng zip, không bấm nộp.


## 1 · Môi trường

In [1]:
import os, subprocess, sys, shutil, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

# PYTHONPATH=src là BẮT BUỘC: subprocess không thừa hưởng sys.path của notebook,
# nên `python3 -m smart_medic.…` sẽ không tìm thấy package nếu thiếu dòng này.
ENV = {**os.environ, "PYTHONPATH": str(ROOT / "src")}

def sh(cmd, *, must=True, mark=True):
    """Chạy lệnh ở gốc repo. must=True ⇒ ô dừng nếu lệnh fail."""
    p = subprocess.run(cmd, shell=True, cwd=ROOT, text=True, env=ENV,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout.rstrip()[:4000])
    if mark:
        print(("✓ " if p.returncode == 0 else "✗ ") + cmd)
    if must and p.returncode != 0:
        raise SystemExit(f"DỪNG — lệnh thất bại: {cmd}")
    return p.returncode, p.stdout

print("repo :", ROOT)
print("python:", sys.version.split()[0])
sh("git rev-parse --short HEAD && git status --porcelain | wc -l | xargs echo 'file bẩn:'")

repo : /Users/handonn/Workplace/AI/smart-medic
python: 3.13.9
fcf9e8a
file bẩn: 4
✓ git rev-parse --short HEAD && git status --porcelain | wc -l | xargs echo 'file bẩn:'


(0, 'fcf9e8a\nfile bẩn: 4\n')

## 2 · CỔNG TOÀN VẸN — không được bỏ

Điểm tính trên offset đã lệch còn **tệ hơn không có điểm**: sai offset cho 0 điểm và
**hoàn toàn im lặng** (20/100 file test không ở NFC; chuẩn hoá trước khi tính offset làm lệch
tới 143 ký tự).

`test_silver_offsets` **FAIL là bình thường** — 165 vi phạm có thật trong corpus bạc, đã biết,
đã lọc lúc nạp. Mọi test khác phải xanh.

In [2]:
rc, out = sh("python3 -m pytest tests/ -q", must=False, mark=False)
fails = [l for l in out.splitlines() if l.startswith("FAILED")]
unexpected = [l for l in fails if "test_silver_offsets" not in l]
if unexpected:
    raise SystemExit("DỪNG — fail ngoài dự kiến:\n" + "\n".join(unexpected))
print("\n✓ cổng toàn vẹn: chỉ còn test_silver_offsets (đã biết)")

........................................................................ [ 52%]
.............................F....................................       [100%]
=================================== FAILURES ===================================
_____________________________ test_silver_offsets ______________________________

    def test_silver_offsets():
        """Generated training data must satisfy the same offset invariant."""
        pairs = list(_silver_pairs())
        if not pairs:
            pytest.skip("no silver corpus present")
    
        all_errs: list[str] = []
        bad_files = 0
        for kind, ann, txt in pairs:
            try:
                entities = json.loads(ann.read_text(encoding="utf-8"))
            except json.JSONDecodeError as exc:
                all_errs.append(f"{kind}/{ann.name}: invalid JSON — {exc}")
                bad_files += 1
                continue
            errs = check_entities(entities, read_raw(txt), f"{kind}/{ann.name}")
          

## 3 · Dựng chỉ mục KB

`data/knowledge_base/` là thư mục **phẳng**; đường dẫn và cách parse RRF nằm ở
`scripts/kb_sources.py`. Chỉ mục là cache — xoá đi thì lệnh này dựng lại.

In [3]:
sh("python3 -c \"import sys; sys.path.insert(0,'scripts'); import kb_sources as k;"
   " k.require(k.ICD10_VI, k.ICD10CM_EN, k.RXNCONSO, k.RXNREL, k.RXNSTY, k.RXNATOMARCHIVE);"
   " print('✓ 6 file KB có mặt')\"")
sh("python3 scripts/annotation_qa/kb.py build")

# Gazetteer của làn R (P1). extract/aho.py và linking/ (P5) đọc CÙNG artifact này —
# dựng một lần, hai chỗ đọc. src/ KHÔNG BAO GIỜ import scripts/, nên file JSON là
# toàn bộ giao diện giữa hai bên.
sh("python3 scripts/build_gazetteer.py --out data/artifacts/gazetteer.json")

✓ 6 file KB có mặt
✓ python3 -c "import sys; sys.path.insert(0,'scripts'); import kb_sources as k; k.require(k.ICD10_VI, k.ICD10CM_EN, k.RXNCONSO, k.RXNREL, k.RXNSTY, k.RXNATOMARCHIVE); print('✓ 6 file KB có mặt')"


built: icd 13189 codes / 14240 names; icd_en 74879; rx 70245 cuis / 118118 names; b2i 96495
✓ python3 scripts/annotation_qa/kb.py build


wrote data/artifacts/gazetteer.json

source          raw seen  kept (pre-merge)   won key
icd_vi             36689             14270     14270
rxnorm             22132             22131     22131
silver             12269              2987      1853
TOTAL                                          38254

usage_overrides_vocabulary: type vote of rxnorm suppressed on 12 key(s) where silver says lab
stoplist drops (key-level, pre-merge): {'literal': 198, 'pattern': 1053, 'short': 22}
  short: ['5', '6', '3', '6', '2', '7', '5', '6', '8', '7', '8', '6', '2', '5', '7']
  literal: ['tin', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'thuốc']
  pattern: ['**********', '*********', '********', '*********', '12.5', '****************', '*******************', '12.5', '9.3', '********', '***********', '90-92%', '***********', '*******', '*******']
dropped by dominan

(0,
 "wrote data/artifacts/gazetteer.json\n\nsource          raw seen  kept (pre-merge)   won key\nicd_vi             36689             14270     14270\nrxnorm             22132             22131     22131\nsilver             12269              2987      1853\nTOTAL                                          38254\n\nusage_overrides_vocabulary: type vote of rxnorm suppressed on 12 key(s) where silver says lab\nstoplist drops (key-level, pre-merge): {'literal': 198, 'pattern': 1053, 'short': 22}\n  short: ['5', '6', '3', '6', '2', '7', '5', '6', '8', '7', '8', '6', '2', '5', '7']\n  literal: ['tin', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'xét nghiệm', 'thuốc']\n  pattern: ['**********', '*********', '********', '*********', '12.5', '****************', '*******************', '12.5', '9.3', '********', '***********', '90-92%', '***********', '*******', '*******']\

## 4 · Chạy trên **gold** → chấm điểm  ·  ĐO

Hai tập chạy khác nhau, đừng lẫn: **gold** có nhãn nên chấm được (đây là ô đo);
**`data/test/`** không có nhãn, nó là bài nộp (ô 5).

Số chính thức nội bộ là **một cặp** `(penalised/greedy_iou, penalised/overlap_type)`
(ADR 0002, cập nhật 30/07). Scorer in **cả ba** cách đọc × **bốn** chế độ căn chỉnh trong một
lần chạy — phải đọc hết:

- **`greedy_iou`** — số chính thức. ⚠ Nó **không so trường `type`** ở đâu cả.
- **`overlap_type`** — **cột chặn**. Sai type 10% mất **12,35** điểm ở đây trong khi
  `greedy_iou` báo 0,00. Thay đổi chỉ được nhận nếu không làm cột này giảm quá **0,010**.
- **`exact`** — **đèn báo bug offset**. Sụt về gần 0 mà hai cột kia bình thường ⇒ bug offset,
  không phải khoảng trống mô hình.
- **`matched`** suy biến — xoá 30% dự đoán của chính mình làm nó *tăng*. Chỉ dùng làm trần.

Ba thứ **bắt buộc báo kèm** mỗi lần chạy (plan-v4 tab 05 §A.2), P2 đã đưa vào ô này:

1. **Bảng lát cắt** kèm `n` và MDE từng lát — lát nào MDE > delta đang xét thì **gạch chân**
   và không được dùng làm bằng chứng. `pho_bien`/`hoi_dap` chỉ 12 tài liệu.
2. **Paired bootstrap** — bar nghiệm thu là `Δ > max(0,010 ; 1,96·SE)` **và** CI95 không chứa
   0. Sàn 0,010 chỉ hợp lệ cho cặp hệ tương quan cao; hai hệ bỏ sót ở chỗ khác nhau không
   phân giải được dưới ~0,84 điểm.
3. **Mật độ entity/file** — đầu vào của `emit_threshold`, không phải số trang trí.


In [4]:
GOLD_TXT = "data/generated_medical_records/restyled/text"
GOLD     = "data/generated_medical_records/restyled/annotations_gold"
PRED_GOLD = "runs/_pred_gold"

# --any-name là BẮT BUỘC ở đây: file gold tên `mtsamples_cardio_0001_pho_bien.txt`,
# không phải `1.txt`. Thiếu cờ này thì loader không thấy file nào và ô báo lỗi.
sh(f"python3 -m smart_medic.cli run --input {GOLD_TXT} --output {PRED_GOLD} --any-name")
sh(f"python3 -m smart_medic.eval.scoring --pred {PRED_GOLD} --gold {GOLD}")

# Cổng nghiệm thu của P1, mỗi tiêu chí in kèm con số quyết định nó.
sh("python3 scripts/analysis/p1_acceptance.py", must=False)

# ── P2 · hạ tầng đo ────────────────────────────────────────────────────────
# Không sinh điểm. Quyết định điểm nào được TIN.

# (a) Cột chặn thành lỗi build. Test này chứng minh nó chặn được một thay đổi
#     xấu biết trước: khôi phục 25% entity với type sai cho greedy_iou +17,17
#     trong khi overlap_type −9,65 — trông như cải tiến lớn nhất dự án.
sh("python3 -m pytest tests/test_alignment_parity.py -q")

# (b) Bảng lát cắt. `n` và MDE đứng cạnh mọi con số; lát nào MDE > delta đang xét
#     thì gạch chân. --types nhân 5 số dòng (thể loại × loại entity).
sh(f"python3 -m smart_medic.eval.slices --pred {PRED_GOLD} --gold {GOLD}"
   f" --types --json runs/p2/slices_gold.json")

# (c) Bốn ca hiệu chuẩn bootstrap. In Δ/SE đo được CẠNH số công bố trong plan —
#     lệch nghĩa là gold hoặc bộ tạo nhiễu đã đổi, không phải bootstrap sai.
sh(f"python3 -m smart_medic.eval.bootstrap --gold {GOLD} --calibrate"
   f" --json runs/p2/bootstrap_calibration.json")


extract    : 162 docs · 7102 candidate spans (43.84/file) · lanes: aho 5675 · kvspan 112 · labvalues 2892 · 1577 dropped as overlapping
             0.6s, no checkpoint loaded
decision   : emit_threshold p=0.25 (branch '<0.50') from candidate density 43.84/file, ratio 0.955
⚠ REGIME MISMATCH — candidate density 43.84/file is ratio 0.955 of gold's 45.9, which the table maps to branch '>0.80', not the '<0.50' branch this P1 constant gate implements.
  The plan's premise was the 15.8/file baseline (ratio 0.34). Lane R is denser than that, so p=0.25 is now a LOOSER gate than the table intends and the run is over-generating rather than under-
  generating. +10% spurious costs 6.10 points, +30% costs 15.61. The three-tier schedule is P6 by design (see the module docstring); this is the number that says P6 has become load-bearing.
validate   : 162 files · 7102 entities · density 43.84/file · 7102/7102 entities · lab assertions cleared 0 · uncodeable candidates cleared 0 · unknown codes droppe


── OFFICIAL SCORE, all three readings ── n=162 documents
aggregation   align             text   assert    cand    final    /100
matched       overlap_type    0.8929   0.7237  0.8556   0.8272   82.72
matched       greedy_iou      0.8881   0.7270  0.8311   0.8170   81.70  (ignores type)
matched       exact           1.0000   0.7288  0.8508   0.8590   85.90  (offset alarm)
penalised     overlap_type    0.6210   0.4995  0.6148   0.5821   58.21  <<< OFFICIAL
penalised     greedy_iou      0.6732   0.5469  0.6492   0.6257   62.57  (ignores type)
penalised     exact           0.5488   0.4016  0.4880   0.4803   48.03  (offset alarm)
docbag        overlap_type    0.6533   0.0926  0.4170   0.3906   39.06

  config hash 2828c775d9b9   missing=1394  spurious=1061
  candidates unweighted ('plain'): 0.5891  → the W_i weighting moves it +0.0257
  text with 1−WER clamped at 0: 0.6260  → documents pushed negative by unbounded WER cost +0.0049  (spec does NOT clamp)

── DIAGNOSTICS ──
type              


lane R  : 162 docs · 7102 candidate spans (43.84/file) · lanes: aho 5675 · kvspan 112 · labvalues 2892 · 1577 dropped as overlapping
gate    : p=0.25

── FOUR READINGS ─────────────────────────────────────────────────
  overlap_type (OFFICIAL)     51.61/100
  greedy_iou   (type-blind)   56.31/100   → type costs +4.70
  exact        (OFFSET BUG)   43.29/100
  matched      (ceiling)      73.47/100

  missing 1394   spurious 1061   density 43.84 entities/file (gold 45.9, ratio 0.955)

── ACCEPTANCE ────────────────────────────────────────────────────
  [PASS] lab types: recall >= 0.70 and precision >= 0.80      recall 0.809  precision 0.879  (gold 3098, pred 2851)
  [PASS] aho.py span recall alone >= 0.30                     0.514  (3824/7435)
  [PASS] gold penalised/overlap_type >= 14.5   << HARD GATE   51.61
  [PASS] 0 nested spans in output                             0 found
  [PASS] raw[s:e] == text on every span                       0 violations
  [PASS] schema clean (incl. lab as

.......                                                                  [100%]
7 passed in 2.89s
✓ python3 -m pytest tests/test_alignment_parity.py -q



── BẢNG LÁT CẮT ── pred=runs/_pred_gold  gold=data/generated_medical_records/restyled/annotations_gold  n=162 tài liệu
   penalised · Δ đang xét = 0.010
   MDE_ref = 1,96·SE·√(N/n), SE=0.054 đo trên cặp tương quan cao nhất có thật (bỏ cờ isFamily) tại N=162.
   Đây là SÀN TỐT NHẤT. Hai hệ bỏ sót ở chỗ khác nhau rộng hơn ~8 lần.

trục        lát           kiểu                 n_doc n_gold   greedy  ovl_type   exact     MDE
──────────────────────────────────────────────────────────────────────────────────────────────
toàn corpus tất cả        mọi loại               162   7435    62.57     58.21   48.03   0.105  ‡
toàn corpus tất cả        TRIỆU_CHỨNG            162   1849    50.58     50.58   41.69   0.105  ‡
toàn corpus tất cả        TÊN_XÉT_NGHIỆM         162   1748    77.19     77.19   70.59   0.105  ‡
toàn corpus tất cả        KẾT_QUẢ_XÉT_NGHIỆM     162   1350    59.23     59.23   49.52   0.105  ‡
toàn corpus tất cả        CHẨN_ĐOÁN              162   1536    43.09     43.09   39.49


── HIỆU CHUẨN BOOTSTRAP ── gold=data/generated_medical_records/restyled/annotations_gold  B=10,000
so sánh                        Δ đo   Δ plan   SE đo  SE plan                  CI95     MDE
bỏ cờ isFamily               -0.261   -0.261   0.053    0.053      [-0.371; -0.163]   0.104
sai 30% mã                   -3.922   -3.922   0.196    0.196      [-4.314; -3.549]   0.383
bỏ 10% entity               -10.164  -10.164   0.378    0.378     [-10.905; -9.430]   0.740
bỏ 10%: seedA vs seedB       -0.067   -0.067   0.531    0.531      [-1.133; +0.988]   1.041

  ✓ cả bốn ca khớp bảng plan-v4 tab 05 §A.4

  Hàng cuối là điều quan trọng nhất: hai hệ bỏ sót ở CHỖ KHÁC NHAU không phân giải được dưới ~1.04 điểm.

  → runs/p2/bootstrap_calibration.json
✓ python3 -m smart_medic.eval.bootstrap --gold data/generated_medical_records/restyled/annotations_gold --calibrate --json runs/p2/bootstrap_calibration.json


(0,
 '\n── HIỆU CHUẨN BOOTSTRAP ── gold=data/generated_medical_records/restyled/annotations_gold  B=10,000\nso sánh                        Δ đo   Δ plan   SE đo  SE plan                  CI95     MDE\nbỏ cờ isFamily               -0.261   -0.261   0.053    0.053      [-0.371; -0.163]   0.104\nsai 30% mã                   -3.922   -3.922   0.196    0.196      [-4.314; -3.549]   0.383\nbỏ 10% entity               -10.164  -10.164   0.378    0.378     [-10.905; -9.430]   0.740\nbỏ 10%: seedA vs seedB       -0.067   -0.067   0.531    0.531      [-1.133; +0.988]   1.041\n\n  ✓ cả bốn ca khớp bảng plan-v4 tab 05 §A.4\n\n  Hàng cuối là điều quan trọng nhất: hai hệ bỏ sót ở CHỖ KHÁC NHAU không phân giải được dưới ~1.04 điểm.\n\n  → runs/p2/bootstrap_calibration.json\n')

## 5 · Chạy trên `data/test/` → `data/output/`  ·  NỘP

`data/test/` **bất biến** — hook chặn ghi, manifest sha256 là lớp hai.
`position` **luôn** index vào chuỗi gốc chưa chuẩn hoá.

In [5]:
CLI = "python3 -m smart_medic.cli run --input data/test --output data/output"
sh(CLI)
sh("python3 -m smart_medic.eval.scoring --pred data/output --describe", must=False)

extract    : 100 docs · 2571 candidate spans (25.71/file) · lanes: aho 2278 · kvspan 47 · labvalues 622 · 376 dropped as overlapping
             0.4s, no checkpoint loaded
decision   : emit_threshold p=0.25 (branch '<0.50') from candidate density 25.71/file, ratio 0.560
⚠ REGIME MISMATCH — candidate density 25.71/file is ratio 0.560 of gold's 45.9, which the table maps to branch '0.50-0.80', not the '<0.50' branch this P1 constant gate implements.
  The plan's premise was the 15.8/file baseline (ratio 0.34). Lane R is denser than that, so p=0.25 is now a LOOSER gate than the table intends and the run is over-generating rather than under-
  generating. +10% spurious costs 6.10 points, +30% costs 15.61. The three-tier schedule is P6 by design (see the module docstring); this is the number that says P6 has become load-bearing.
validate   : 100 files · 2571 entities · density 25.71/file · 2571/2571 entities · lab assertions cleared 0 · uncodeable candidates cleared 0 · unknown codes dropp

(0,
 '\n── STRUCTURE of data/output ──\n  documents            100\n  entities             2571  (25.7 per doc)\n  type distribution\n      TRIỆU_CHỨNG               959   37.3%\n      CHẨN_ĐOÁN                 787   30.6%\n      TÊN_XÉT_NGHIỆM            479   18.6%\n      THUỐC                     184    7.2%\n      KẾT_QUẢ_XÉT_NGHIỆM        162    6.3%\n  span length (words)  median 2  mean 2.05  max 8\n  codes/entity (CĐ+thuốc) {0: 23, 1: 940, 2: 8}\n  assertions/entity       {0: 2571}\n\n(no --gold given, so no score. Hand-label a dev set and pass --gold.)\n')

## 6 · Đóng gói `output.zip` (+ ba zip probe)

Đúng **100** file `1.json` … `100.json` trong thư mục `output/`, **0 file phụ**.
`unzip -l` phải được **dán nguyên** vào báo cáo — không mô tả bằng lời.

Khối thứ hai dựng **ba biến thể probe của P2** từ đúng `data/output/` đó, mỗi biến thể qua
cùng 7 kiểm tra đóng gói:

| zip | biến | câu hỏi nó trả lời |
|---|---|---|
| `output_probe_A.zip` | span + type, mọi trường khác rỗng | recall span thật trên test = `A / 51,55` |
| `output_probe_Aprime.zip` | A, **đảo 100% type**, span y nguyên | BTC có so `type` khi căn chỉnh không? |
| `output_probe_B.zip` | A + mã mức `IN` cho THUỐC | gold BTC ở tầng `IN` hay `SCD`? (ADR 0001) |

Thứ tự nộp **bắt buộc** A → A′ → B, **mỗi lần một biến**. Câu hỏi và delta kỳ vọng của cả ba
đã ghi **trước** ở [`runs/p2/SUBMISSION_LOG.md`](../runs/p2/SUBMISSION_LOG.md).

> ⚠ `git status` bẩn ⇒ manifest mang `git_dirty: true` và bộ đóng gói in
> *"must NOT be submitted"*. **Commit rồi dựng lại** trước khi nộp.
> Và notebook **không bấm nộp** — đó là quyết định của con người, 5 lần/ngày.


In [6]:
PKG = "python3 scripts/submit/package_submission.py"
if not (ROOT / "scripts/submit/package_submission.py").exists():
    print("⏳ CHƯA CÓ — P0 còn ⬜. Lệnh sẽ là:\n   " + PKG)
else:
    sh(PKG)
    sh("unzip -l output.zip")

# ── P2 · ba zip probe ──────────────────────────────────────────────────────
# `Aprime` chứ không phải `A'`: dấu nháy phải escape ở mọi chỗ nó xuất hiện —
# dòng lệnh, tên thư mục, trường manifest. probe.py nhận cả hai (ALIASES).
if not (ROOT / "src/smart_medic/eval/probe.py").exists():
    print("⏳ CHƯA CÓ — P2 còn ⬜.")
else:
    for v in ("A", "Aprime", "B"):
        sh(f"python3 -m smart_medic.eval.probe --pred data/output --variant {v}"
           f" --out runs/p2/pred_{v} --json runs/p2/build_{v}.json")
        sh(f"python3 scripts/submit/package_submission.py --pred runs/p2/pred_{v}"
           f" --probe {v} --run-dir runs/p2/pkg_{v} --out runs/p2/output_probe_{v}.zip"
           f" --no-freeze")

    # Delta kỳ vọng NỘI BỘ — ghi TRƯỚC khi nộp, nếu không thì kết quả thật không
    # có gì để sai so với nó. Chạy trên gold, nơi ta còn nhãn để chấm.
    sh(f"python3 -m smart_medic.eval.probe --expect --gold {GOLD} --pred {PRED_GOLD}"
       f" --json runs/p2/expect_real.json")
    print("\n→ bảng câu hỏi / delta kỳ vọng: runs/p2/SUBMISSION_LOG.md")
    print("  NGƯỜI bấm nộp, không phải notebook. Thứ tự A → Aprime → B.")


run        : runs/2026-07-30T15-03_fcf9e8a
predictions: data/output  ->  staging
  100 files · 2571 entities · density 25.71/file · 2571/2571 entities · lab assertions cleared 0 · uncodeable candidates cleared 0 · unknown codes dropped 0 · dropped malformed 0 / bad type 0 / nested 0 / duplicate 0

✓ 7/7 archive checks passed
  output.zip  (71,247 bytes)
  runs/2026-07-30T15-03_fcf9e8a/manifest.json

  !! WORKING TREE IS DIRTY — this build must NOT be submitted.
     runs/README.md: a dirty tree means the run cannot be reproduced.
✓ python3 scripts/submit/package_submission.py
Archive:  output.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
     6640  01-01-1980 00:00   output/1.json
     8125  01-01-1980 00:00   output/2.json
     8634  01-01-1980 00:00   output/3.json
    13600  01-01-1980 00:00   output/4.json
     4602  01-01-1980 00:00   output/5.json
     5407  01-01-1980 00:00   output/6.json
     3969  01-01-1980 00:00   output/7.json
     7974  01-01-1

run        : runs/p2/pkg_A
predictions: runs/p2/pred_A  ->  staging
  100 files · 2571 entities · density 25.71/file · 2571/2571 entities · lab assertions cleared 0 · uncodeable candidates cleared 0 · unknown codes dropped 0 · dropped malformed 0 / bad type 0 / nested 0 / duplicate 0

✓ 7/7 archive checks passed
  runs/p2/output_probe_A.zip  (66,781 bytes)
  runs/p2/pkg_A/manifest.json

  !! WORKING TREE IS DIRTY — this build must NOT be submitted.
     runs/README.md: a dirty tree means the run cannot be reproduced.
✓ python3 scripts/submit/package_submission.py --pred runs/p2/pred_A --probe A --run-dir runs/p2/pkg_A --out runs/p2/output_probe_A.zip --no-freeze

── PROBE A' ── data/output → runs/p2/pred_Aprime
  100 tài liệu · 2571 entity · mật độ 25.71/file
  type đảo: 2571/2571 (100% — spans byte-identical với A)

  Đóng gói:  python3 scripts/submit/package_submission.py --pred runs/p2/pred_Aprime --probe Aprime
  KHÔNG tự nộp. Nộp bài là quyết định của con người.

  → runs/p2/build

run        : runs/p2/pkg_Aprime
predictions: runs/p2/pred_Aprime  ->  staging
  100 files · 2571 entities · density 25.71/file · 2571/2571 entities · lab assertions cleared 0 · uncodeable candidates cleared 0 · unknown codes dropped 0 · dropped malformed 0 / bad type 0 / nested 0 / duplicate 0

✓ 7/7 archive checks passed
  runs/p2/output_probe_Aprime.zip  (66,702 bytes)
  runs/p2/pkg_Aprime/manifest.json

  !! WORKING TREE IS DIRTY — this build must NOT be submitted.
     runs/README.md: a dirty tree means the run cannot be reproduced.
✓ python3 scripts/submit/package_submission.py --pred runs/p2/pred_Aprime --probe Aprime --run-dir runs/p2/pkg_Aprime --out runs/p2/output_probe_Aprime.zip --no-freeze

── PROBE B ── data/output → runs/p2/pred_B
  100 tài liệu · 2571 entity · mật độ 25.71/file
  THUỐC có mã: 181/184 (98.4%) · 213 mã mức IN

  Đóng gói:  python3 scripts/submit/package_submission.py --pred runs/p2/pred_B --probe B
  KHÔNG tự nộp. Nộp bài là quyết định của con người.

  → 

run        : runs/p2/pkg_B
predictions: runs/p2/pred_B  ->  staging
  100 files · 2571 entities · density 25.71/file · 2571/2571 entities · lab assertions cleared 0 · uncodeable candidates cleared 0 · unknown codes dropped 0 · dropped malformed 0 / bad type 0 / nested 0 / duplicate 0

✓ 7/7 archive checks passed
  runs/p2/output_probe_B.zip  (68,059 bytes)
  runs/p2/pkg_B/manifest.json

  !! WORKING TREE IS DIRTY — this build must NOT be submitted.
     runs/README.md: a dirty tree means the run cannot be reproduced.
✓ python3 scripts/submit/package_submission.py --pred runs/p2/pred_B --probe B --run-dir runs/p2/pkg_B --out runs/p2/output_probe_B.zip --no-freeze



── DELTA KỲ VỌNG (NỘI BỘ) ── gold=data/generated_medical_records/restyled/annotations_gold  pred=runs/_pred_gold  n=162
biến thể    greedy_iou  overlap_type    exact   Δ vs A (greedy)   Δ vs A (ovl_type)
A                56.31         51.61    43.29                 —                   —
A'               56.31          0.40     0.23             +0.00              -51.21
B                59.26         54.53    45.03             +2.96               +2.92
  A' vs A (greedy_iou, paired B=10.000): Δ=-51.211  SE=0.984  CI95=[-53.157; -49.269]
  B vs A (greedy_iou, paired B=10.000): Δ=+2.922  SE=0.226  CI95=[+2.489; +3.378]
  B mã hoá 856/986 span THUỐC (86.8%), 1198 mã

  → runs/p2/expect_real.json
✓ python3 -m smart_medic.eval.probe --expect --gold data/generated_medical_records/restyled/annotations_gold --pred runs/_pred_gold --json runs/p2/expect_real.json

→ bảng câu hỏi / delta kỳ vọng: runs/p2/SUBMISSION_LOG.md
  NGƯỜI bấm nộp, không phải notebook. Thứ tự A → Aprime → B.


## 7 · Diễn tập tái lập — **chống bị loại**

Rủi ro duy nhất **không mua lại được bằng điểm**: top ~15 đội nộp source + weights, ban tổ chức
chạy lại trên private test, **cài không được ⇒ bị loại**.

Chạy **hai lần**: lần đầu để tìm lỗi khi còn thời gian sửa, lần hai để xác nhận.
Diễn tập *sau* khi nộp là vô ích. Temperature 0 **không** cho tính xác định — nếu không
bit-identical thì cố định `batch_size=1`, còn lệch nữa thì đóng gói cache đầu ra kèm theo.

In [7]:
print("""Chạy NGOÀI notebook này, trên container sạch:

  docker run --rm -it -v "$PWD":/w -w /w python:3.13-slim bash -lc '
    pip install --require-hashes -r requirements.lock &&
    python3 -m smart_medic.cli run --input data/test --output /tmp/out &&
    python3 scripts/submit/package_submission.py --out /tmp/out.zip --run-dir /tmp/run --no-run-record'

  # rồi so với bản đã nộp. Bộ ghi archive là TẤT ĐỊNH (ADR 0005), nên cmp phải
  # SẠCH TUYỆT ĐỐI — một byte lệch là một dự đoán lệch, không phải nhiễu đóng gói.
  cmp /tmp/out.zip runs/<ts>_<sha>/output.zip && echo "✓ tái lập được"
""")
if (ROOT / "runs").exists():
    sh("ls -1 runs/ | tail -5", must=False)

Chạy NGOÀI notebook này, trên container sạch:

  docker run --rm -it -v "$PWD":/w -w /w python:3.13-slim bash -lc '
    pip install --require-hashes -r requirements.lock &&
    python3 -m smart_medic.cli run --input data/test --output /tmp/out &&
    python3 scripts/submit/package_submission.py --out /tmp/out.zip --run-dir /tmp/run --no-run-record'

  # rồi so với bản đã nộp. Bộ ghi archive là TẤT ĐỊNH (ADR 0005), nên cmp phải
  # SẠCH TUYỆT ĐỐI — một byte lệch là một dự đoán lệch, không phải nhiễu đóng gói.
  cmp /tmp/out.zip runs/<ts>_<sha>/output.zip && echo "✓ tái lập được"

_p5_lift_only
_p5_nolift
_pred_gold
leverage_map.json
p2
✓ ls -1 runs/ | tail -5


## 8 · TỰ KIỂM — chạy sau **mỗi** phase

Notebook này nhân bản các lệnh của repo, nên nó **trôi được**. Ô dưới đây bắt chỗ trôi:
mọi đường dẫn nó nhắc tới phải tồn tại (hoặc là artifact P0–P7 đã biết là chưa có), và mọi
cờ dòng lệnh nó dùng phải thật sự có trong `--help`.

**Một phase chỉ được coi là xong khi ô này xanh.** Nếu đỏ: sửa notebook, đừng sửa cách đọc.

In [8]:
import re, json

NB = ROOT / "notebooks/runbook.ipynb"
cells = json.loads(NB.read_text())["cells"] if NB.exists() else []
src = "".join("".join(c["source"]) for c in cells if c["cell_type"] == "code")

PENDING = {  # artifact chưa có, thuộc phase nào — KHÔNG tính là lệch
    "requirements.lock": "P7",
}
REQUIRED = [
    "scripts/kb_sources.py", "scripts/annotation_qa/kb.py",
    "scripts/analysis/measure_data.py", "scripts/analysis/leverage_map.py",
    "src/smart_medic/eval/scoring.py", "tests/test_offsets.py",
    "data/generated_medical_records/restyled/text",
    "data/generated_medical_records/restyled/annotations_gold",
    "data/test", ".claude/prompts/p0_prompt.md",
    # P0 · nền móng
    "configs/pipeline.yaml", "configs/models.yaml", "configs/metric.yaml",
    "src/smart_medic/io/document.py", "src/smart_medic/io/corpus.py",
    "src/smart_medic/layout/lines.py", "src/smart_medic/layout/outline.py",
    "src/smart_medic/layout/kv.py",
    "src/smart_medic/validate/schema.py", "src/smart_medic/validate/offsets.py",
    "src/smart_medic/validate/emit_json.py",
    "scripts/submit/package_submission.py",
    "tests/test_no_api_in_runtime.py", "tests/test_layer_boundaries.py",
    "pyproject.toml", "Makefile",
    # P1 · làn R, sàn recall không model
    "src/smart_medic/cli.py",
    "src/smart_medic/extract/spans.py", "src/smart_medic/extract/aho.py",
    "src/smart_medic/extract/labvalues.py", "src/smart_medic/extract/kvspan.py",
    "src/smart_medic/decision/emit.py",
    "resources/lab_patterns.yaml", "resources/gazetteer_vi.yaml",
    "resources/section_type_prior.yaml",   # P3 · prior theo section
    "scripts/build_gazetteer.py",
    "scripts/analysis/p1_acceptance.py", "scripts/analysis/sweep_recall_floor.py",
    "tests/test_extract.py", "tests/test_decision.py",
    # P2 · hiệu chuẩn hộp đen
    "src/smart_medic/eval/probe.py", "src/smart_medic/eval/slices.py",
    "src/smart_medic/eval/bootstrap.py", "tests/test_alignment_parity.py",
    "docs/decisions/0001-drug-tty.md", "docs/decisions/0002-metric-reading.md",
    "runs/p2/SUBMISSION_LOG.md",
    # P5 · candidates
    "src/smart_medic/linking/icd.py", "src/smart_medic/linking/rxnorm.py",
    "src/smart_medic/linking/edge_verify.py",
    "tests/test_linking.py",
]

bad = [f"THIẾU (đáng lẽ phải có): {f}" for f in REQUIRED if not (ROOT / f).exists()]
for f, ph in PENDING.items():
    print(("✓ đã có    " if (ROOT / f).exists() else f"⏳ chờ {ph:7}") + f)

# `data/artifacts/` là cache SINH LẠI ĐƯỢC (gitignore) — thiếu thì ô 3 dựng lại,
# nên nó không phải "lệch", nhưng ô 4/5 sẽ chết nếu chạy mà chưa có.
GAZ = ROOT / "data/artifacts/gazetteer.json"
print(("✓ đã có    " if GAZ.exists() else "⚠ chạy ô 3 ") + "data/artifacts/gazetteer.json")

# Ba zip probe của P2. Không phải "lệch" nếu thiếu — ô 6 dựng lại — nhưng nếu
# thiếu thì KHÔNG có gì để nộp, và đó là toàn bộ đầu ra của phase P2.
for v in ("A", "Aprime", "B"):
    z = ROOT / f"runs/p2/output_probe_{v}.zip"
    print(("✓ đã có    " if z.exists() else "⚠ chạy ô 6 ") + f"runs/p2/output_probe_{v}.zip")

# Cờ dòng lệnh: chỉ soi các LỆNH THẬT, không soi chính ô kiểm tra này (các mẫu
# dưới đây viết `\.` nên chúng không tự khớp với chính mình).
def check_flags(cmd_help, pattern, label):
    """Mọi cờ notebook dùng cho một lệnh phải thật sự có trong --help của lệnh đó."""
    cmds = re.findall(pattern, src)
    used = {f for c in cmds for f in re.findall(r"--[a-z][a-z-]+", c)}
    for flag in sorted(used):
        if flag not in cmd_help:
            bad.append(f"NOTEBOOK DÙNG CỜ KHÔNG TỒN TẠI: {flag}  ({label})")
    return used

_, helptxt = sh("python3 -m smart_medic.eval.scoring --help", must=False, mark=False)
used = check_flags(helptxt, r'"[^"]*smart_medic\.eval\.scoring[^"]*"', "scoring.py")

# Cờ của BỘ ĐÓNG GÓI cũng phải thật. Ô 7 nhắc một lệnh chạy trong container, và
# một cờ sai ở đó chỉ lộ ra giữa buổi diễn tập tái lập — muộn nhất có thể.
# Mẫu chạy tới dấu `)` chứ không tới dấu nháy: lệnh đóng gói probe ở ô 6 trải
# trên nhiều đoạn f-string, dừng ở nháy thì bỏ sót quá nửa số cờ.
pkgused = set()
if (ROOT / "scripts/submit/package_submission.py").exists():
    _, pkghelp = sh("python3 scripts/submit/package_submission.py --help",
                    must=False, mark=False)
    pkgused = check_flags(pkghelp, r"package_submission\.py[^)]*",
                          "package_submission.py")

# Cờ của CLI SUY LUẬN. Ô 4 phải có --any-name (file gold không đánh số) và ô 7 nhắc
# lại lệnh trong container; một cờ sai ở đây làm cả bài nộp không dựng được.
cliused = set()
if (ROOT / "src/smart_medic/cli.py").exists():
    _, clihelp = sh("python3 -m smart_medic.cli run --help", must=False, mark=False)
    cliused = check_flags(clihelp, r"smart_medic\.cli run[^\n\"']*", "cli.py run")
    if "--any-name" not in cliused:
        bad.append("Ô 4 chấm điểm trên gold mà KHÔNG có --any-name: file gold tên "
                   "`mtsamples_*.txt`, loader sẽ không thấy file nào")

# Ba module đo của P2. Cùng lý do như bộ đóng gói: lệnh trải nhiều dòng.
p2used = {}
for mod in ("probe", "slices", "bootstrap"):
    path = ROOT / f"src/smart_medic/eval/{mod}.py"
    if not path.exists():
        continue
    _, h = sh(f"python3 -m smart_medic.eval.{mod} --help", must=False, mark=False)
    p2used[mod] = check_flags(h, rf"smart_medic\.eval\.{mod}[^)]*", f"{mod}.py")

print()
if not bad:
    print("cờ scorer   :", sorted(used) or "—", " → đều hợp lệ")
    print("cờ đóng gói :", sorted(pkgused) or "—", " → đều hợp lệ")
    print("cờ cli      :", sorted(cliused) or "—", " → đều hợp lệ")
    for mod, flags in p2used.items():
        print(f"cờ {mod:<9}:", sorted(flags) or "—", " → đều hợp lệ")

if bad:
    print("✗ NOTEBOOK ĐÃ TRÔI KHỎI REPO:")
    for b in bad: print("   -", b)
    raise SystemExit("Sửa notebooks/runbook.ipynb trước khi coi phase là xong.")
print("✓ notebook còn khớp repo")


⏳ chờ P7     requirements.lock
✓ đã có    data/artifacts/gazetteer.json
✓ đã có    runs/p2/output_probe_A.zip
✓ đã có    runs/p2/output_probe_Aprime.zip
✓ đã có    runs/p2/output_probe_B.zip
usage: scoring.py [-h] [--pred PRED] [--gold GOLD] [--describe]
                  [--cand-formula {official,plain}] [--json]

Internal scorer for the Viettel AI Race 2026 Vòng-1 metric.

options:
  -h, --help            show this help message and exit
  --pred PRED
  --gold GOLD           directory of gold N.json files
  --describe            structural summary only, no gold needed
  --cand-formula {official,plain}
  --json                emit machine-readable JSON
usage: package_submission.py [-h] [--pred PRED] [--source SOURCE] [--out OUT]
                             [--expect EXPECT] [--probe PROBE] [--no-freeze]
                             [--no-kb-check] [--run-dir RUN_DIR]
                             [--no-run-record]

Build `output.zip` and the immutable run record. Build-time, not infere

usage: probe.py [-h] [--pred PRED] [--variant {A,A',Aprime,B,C}] [--out OUT]
                [--assertions-from ASSERTIONS_FROM] [--max-codes MAX_CODES]
                [--expect] [--gold GOLD] [--json JSON]

Probe variants — one submission, one variable, one black box opened.

options:
  -h, --help            show this help message and exit
  --pred PRED           source prediction directory
  --variant {A,A',Aprime,B,C}
  --out OUT             where to write the probe records
  --assertions-from ASSERTIONS_FROM
                        Probe C donor: a prediction dir that carries
                        assertions
  --max-codes MAX_CODES
  --expect              internal expected deltas for A / A' / B, needs --gold
  --gold GOLD
  --json JSON
usage: slices.py [-h] --pred PRED --gold GOLD [--text TEXT]
                 [--baseline BASELINE] [--delta DELTA] [--types] [--json JSON]

The mandatory slice table — every slice carries its `n` and its MDE.

options:
  -h, --help           show 

---

## Xong

`output.zip` + `runs/<ISO8601>_<git-sha7>/{output/, manifest.json, score.json}`,
và ba zip probe của P2 ở `runs/p2/`.
Nộp bài **luôn là quyết định của con người** — 5 lần/ngày, không tự động hoá.
Thứ tự P2 bắt buộc: **A → Aprime → B**, mỗi lần một biến, delta kỳ vọng đã ghi trước ở
[`runs/p2/SUBMISSION_LOG.md`](../runs/p2/SUBMISSION_LOG.md).

**Ba việc không được bỏ dù hết thời gian:** `pytest tests/test_offsets.py` sạch ·
ba test chống rò rỉ API xanh · `unzip -l output.zip` đúng 100 file trong thư mục `output/`.
Ba việc đó không cho thêm điểm nào, nhưng bỏ một trong ba là mất toàn bộ 70,00.

🔁 **Sau mỗi phase:** chạy lại notebook này từ đầu và cập nhật nếu ô 8 báo đỏ. Đây là một mục
trong tiêu chí nghiệm thu của mọi phase, không phải việc dọn dẹp tuỳ hứng.

<sub>Không nằm trên đường ra output, chạy khi cần: `python3 scripts/analysis/measure_data.py`
(số liệu corpus/KB) · `python3 scripts/analysis/leverage_map.py --seeds 6 --extras --json
runs/leverage_map.json` (bản đồ đòn bẩy) · `python3 -m smart_medic.eval.bootstrap --gold GOLD
--a DIR_A --b DIR_B` (so hai lần chạy, có CI).</sub>
